In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
import os
import glob
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import rioxarray as rxr
import xarray as xr
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB
from pyproj import Proj
from datetime import datetime

# ── Constants ──────────────────────────────────────────────────────────────
GSLIB_SENTINEL = 9.966e+20    
UTM_PROJ       = "+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs"
NODATA_OUT     = -9999.0
GRID_RES_KM    = 1.0      

# --- DATE FILTERING SETUP ---
START_DATE = "20010101"  
END_DATE   = "20051231"  

#the first start date is 2001-01-01 and final end data 2020-12-31
#chunk 1 -  2001-01-01 to 2005-12-31 for initial testing phase

# Fallback thresholds from kriging to bilinear resampling ---
WET_PIXEL_THRESHOLD = 1.0  # mm threshold to define a "wet" pixel
WET_PERCENT_MIN     = 0.05 # 5% minimum wet pixels required to trigger Kriging    

seasons = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
model_it    = {'spherical': 1, 'exponential': 2, 'gaussian': 3}
tmin, tmax  = -9999, 9999

# ── Paths ──────────────────────────────────────────────────────────────────
era5L_dir    = r"..\Data\Datasets\ERA5L_prec"
var_dir      = r"..\Data\Processed\Variogram_analysis"
var_params_path = r"..\Data\Processed\Variogram_analysis\variogram_params_singleDomain.csv"
temp_dir     = r"..\Data\Processed\Kriging_Predictors\Temp"
output_dir   = r"..\Data\Processed\Kriging_Predictors\Kriged_rasters"
os.makedirs(temp_dir,   exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [2]:
# ── Helper functions ───────────────────────────────────────────────────────
def get_season(month):
    for s, months in seasons.items():
        if month in months:
            return s
    return None

def coords_to_utm_km(lons, lats):
    proj = Proj(UTM_PROJ)
    x_m, y_m = proj(lons, lats)
    return x_m / 1000.0, y_m / 1000.0

def apply_trend(df, vp, xcol='x_km', ycol='y_km'):
    return (
        vp.coef_x  * df[xcol]           +
        vp.coef_y  * df[ycol]           +
        vp.coef_x2 * df[xcol] ** 2      +
        vp.coef_y2 * df[ycol] ** 2      +
        vp.coef_xy * df[xcol] * df[ycol]+
        vp.intercept
    )

def safe_nscore_lookup(ns_pred, lookup_df):
    lut = lookup_df.sort_values('nscore_value').drop_duplicates('nscore_value')
    return np.interp(ns_pred, lut['nscore_value'].values, lut['original_value'].values)

def mask_gslib_sentinel(arr, threshold=1e10):
    return np.where(np.abs(arr) > threshold, np.nan, arr.astype(float))

# ── Pre-load & Filtering Logic ──────────────────────────────────────────────
all_files = sorted(glob.glob(os.path.join(era5L_dir, "*.tif")))

# Create a filtered list of (date_string, file_path)
processing_queue = []
for f in all_files:
    # Extracts YYYYMMDD based on your -12:-4 convention
    date_str = os.path.basename(f)[-12:-4] 
    if START_DATE <= date_str <= END_DATE:
        processing_queue.append((date_str, f))

if not processing_queue:
    print(f"No files found between {START_DATE} and {END_DATE}.")
else:
    print(f"Queued {len(processing_queue)} files for processing.")

# Setup grid using the first file in the filtered queue
with rxr.open_rasterio(processing_queue[0][1]) as sample:
    sample_grid = sample.squeeze('band', drop=True)
    lon_grid, lat_grid = np.meshgrid(sample_grid.x.values, sample_grid.y.values)
    x_km_all, y_km_all = coords_to_utm_km(lon_grid.ravel(), lat_grid.ravel())

# Load single domain variogram parameters
var_params_all = pd.read_csv(var_params_path).set_index('season')

Queued 33 files for processing.


In [3]:
# %%
log_rows = []

# Now iterating over the (date, path) pairs directly
for date_str, file_path in processing_queue:
    month  = int(date_str[4:6])
    season = get_season(month)
    print(f"\n{'-'*60}\n  Date: {date_str}  |  Season: {season}")

    if season not in var_params_all.index:
        print(f"  [SKIP] No params for season {season}"); continue

    with rxr.open_rasterio(file_path) as era5_rxr:
        era5_arr = era5_rxr.squeeze('band', drop=True).load().values

    # Base dataframe for the entire domain
    base_df = pd.DataFrame({
        'x_km': x_km_all,
        'y_km': y_km_all,
        'era5_prec': era5_arr.ravel().astype(float)
    }).dropna(subset=['era5_prec'])
    base_df = base_df[base_df['era5_prec'] >= 0].reset_index(drop=True)

    if len(base_df) < 4:
        print(f"  [SKIP] Only {len(base_df)} pixels"); continue

    # ── DECISION GATE ─────────────────────────────────────────────────────
    total_valid_pixels = len(base_df)
    wet_pixels         = (base_df['era5_prec'] >= WET_PIXEL_THRESHOLD).sum()
    wet_percentage     = wet_pixels / total_valid_pixels
    final_path         = os.path.join(output_dir, f"kriged_{date_str}.tif")
    utm_path           = os.path.join(temp_dir,   f"utm_{date_str}.tif")

    vp = var_params_all.loc[season].copy()

    # ── DRY DAY FALLBACK: bilinear resample → Gaussian smooth → WGS84 ────
    if wet_percentage < WET_PERCENT_MIN:
        print(f"  [FALLBACK] Dry day: {wet_percentage*100:.1f}% wet pixels. Using Bilinear + Gaussian smooth.")
        log_rows.append({'date': date_str, 'season': season, 'method': 'bilinear_smoothed',
                         'wet_pct': round(wet_percentage * 100, 2)})

        sigma_pixels = 4.5

        with rxr.open_rasterio(file_path) as src:
            src_utm = src.rio.reproject(
                'EPSG:32645',
                resolution=GRID_RES_KM * 1000,
                resampling=Resampling.bilinear
            )
            utm_arr = src_utm.squeeze().values.astype('float32')

            nodata_mask = (utm_arr == NODATA_OUT) | np.isnan(utm_arr)
            fill_arr = utm_arr.copy()
            fill_arr[nodata_mask] = 0.0

            smoothed = gaussian_filter(fill_arr, sigma=sigma_pixels)
            weight_mask = gaussian_filter((~nodata_mask).astype('float32'), sigma=sigma_pixels)

            with np.errstate(invalid='ignore', divide='ignore'):
                smoothed = np.where(weight_mask > 0.1, smoothed / weight_mask, NODATA_OUT)

            smoothed[nodata_mask] = NODATA_OUT
            smoothed = np.clip(smoothed, 0, None).astype('float32')

            utm_da = src_utm.copy(data=smoothed[np.newaxis, ...])
            utm_da.rio.write_nodata(NODATA_OUT, inplace=True)
            utm_da.rio.to_raster(utm_path, dtype='float32')

        with rasterio.open(utm_path) as src:
            t_wgs, w, h = calculate_default_transform(src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
            out_wgs = np.full((1, h, w), NODATA_OUT, dtype='float32')
            reproject(
                source=rasterio.band(src, 1), destination=out_wgs,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=t_wgs, dst_crs='EPSG:4326',
                resampling=Resampling.bilinear,
                src_nodata=NODATA_OUT, dst_nodata=NODATA_OUT
            )
            profile = src.profile.copy()
            profile.update(crs='EPSG:4326', transform=t_wgs, width=w, height=h)
            with rasterio.open(final_path, 'w', **profile) as dst:
                dst.write(out_wgs)

        if os.path.exists(utm_path):
            os.remove(utm_path)
        print(f"  [OK] → {final_path}")
        continue

    # ── WET DAY: KRIGING ──────────────────────────────────────────────────
    print(f"  [KRIGING] {wet_percentage*100:.1f}% wet pixels. Proceeding with Ordinary Kriging.")
    log_rows.append({'date': date_str, 'season': season, 'method': 'kriging',
                     'wet_pct': round(wet_percentage * 100, 2)})

    if vp.trend_removed:
        base_df['residual'] = base_df['era5_prec'] - apply_trend(base_df, vp)
    else:
        base_df['residual'] = base_df['era5_prec']

    if base_df['residual'].std() < 1e-6:
        print("  [SKIP] Zero variance"); continue

    ns_data, t_vals, t_ns = geostats.nscore(base_df, 'residual')
    base_df['ns'] = ns_data
    lookup_df = pd.DataFrame({'original_value': t_vals, 'nscore_value': t_ns})

    h_range = vp['range_km']
    vario = GSLIB.make_variogram(
        nug=vp['nugget'], nst=1, it1=model_it[vp['model_type']],
        cc1=vp['partial_sill'], azi1=vp['azimuth_major'],
        hmaj1=h_range, hmin1=h_range
    )

    x_min, x_max = base_df['x_km'].min(), base_df['x_km'].max()
    y_min, y_max = base_df['y_km'].min(), base_df['y_km'].max()
    xmn, ymn = np.floor(x_min) + GRID_RES_KM/2, np.floor(y_min) + GRID_RES_KM/2
    nx, ny = int(np.ceil((x_max-xmn)/GRID_RES_KM))+1, int(np.ceil((y_max-ymn)/GRID_RES_KM))+1

    kmap, _ = geostats.kb2d(
        df=base_df, xcol='x_km', ycol='y_km', vcol='ns',
        tmin=tmin, tmax=tmax, nx=nx, xmn=xmn, xsiz=GRID_RES_KM,
        ny=ny, ymn=ymn, ysiz=GRID_RES_KM, nxdis=1, nydis=1,
        ndmin=1, ndmax=8, radius=float(h_range), ktype=1, skmean=0, vario=vario
    )

    kmap = mask_gslib_sentinel(kmap)
    if np.isnan(kmap).all():
        print("  [SKIP] No nodes estimated"); continue

    x_nodes = xmn + np.arange(nx) * GRID_RES_KM
    y_nodes = ymn + (ny - 1 - np.arange(ny)) * GRID_RES_KM
    xx, yy  = np.meshgrid(x_nodes, y_nodes)

    out = pd.DataFrame({'x_km': xx.ravel(), 'y_km': yy.ravel(), 'ns_pred': kmap.ravel()}).dropna()
    out['residual_pred'] = safe_nscore_lookup(out['ns_pred'].values, lookup_df)
    out['trend'] = apply_trend(out, vp) if vp.trend_removed else 0.0
    out['prec_pred'] = (out['residual_pred'] + out['trend']).clip(lower=0)

    all_x, all_y = np.sort(out['x_km'].unique()), np.sort(out['y_km'].unique())[::-1]
    grid_pivot = out.pivot(index='y_km', columns='x_km', values='prec_pred').reindex(index=all_y, columns=all_x)
    out_grid = np.where(np.isnan(grid_pivot.values), NODATA_OUT, grid_pivot.values).astype('float32')

    transform_utm = from_origin(
        west=all_x.min()*1000 - (GRID_RES_KM*500), 
        north=all_y.max()*1000 + (GRID_RES_KM*500), 
        xsize=GRID_RES_KM*1000, ysize=GRID_RES_KM*1000
    )

    with rasterio.open(utm_path, 'w', driver='GTiff', height=out_grid.shape[0], width=out_grid.shape[1],
                       count=1, dtype='float32', crs='EPSG:32645', transform=transform_utm, nodata=NODATA_OUT) as dst:
        dst.write(out_grid, 1)

    with rasterio.open(utm_path) as src:
        t_wgs, w, h = calculate_default_transform(src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
        out_wgs = np.full((1, h, w), NODATA_OUT, dtype='float32')
        reproject(
            source=rasterio.band(src, 1), destination=out_wgs, src_transform=src.transform, 
            src_crs=src.crs, dst_transform=t_wgs, dst_crs='EPSG:4326', 
            resampling=Resampling.bilinear, src_nodata=NODATA_OUT, dst_nodata=NODATA_OUT
        )
        profile = src.profile.copy()
        profile.update(crs='EPSG:4326', transform=t_wgs, width=w, height=h)
        with rasterio.open(final_path, 'w', **profile) as dst:
            dst.write(out_wgs)

    if os.path.exists(utm_path):
        os.remove(utm_path)
    print(f"  [OK] → {final_path}")

# Final step: Save logs
if log_rows:
    pd.DataFrame(log_rows).to_csv(os.path.join(output_dir, "processing_summary.csv"), index=False)


------------------------------------------------------------
  Date: 20051129  |  Season: post_monsoon
  [FALLBACK] Dry day: 3.1% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051129.tif

------------------------------------------------------------
  Date: 20051130  |  Season: post_monsoon
  [KRIGING] 7.7% wet pixels. Proceeding with Ordinary Kriging.
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 512/512 [01:08<00:00,  7.48it/s]


  Estimated   437248 blocks 
      average   -0.5691670611615722  variance  0.9921351586355383
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051130.tif

------------------------------------------------------------
  Date: 20051201  |  Season: winter
  [KRIGING] 5.4% wet pixels. Proceeding with Ordinary Kriging.
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 512/512 [01:09<00:00,  7.33it/s]


  Estimated   451584 blocks 
      average   0.3138523010933484  variance  0.40662078622239484
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051201.tif

------------------------------------------------------------
  Date: 20051202  |  Season: winter
  [FALLBACK] Dry day: 2.6% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051202.tif

------------------------------------------------------------
  Date: 20051203  |  Season: winter
  [FALLBACK] Dry day: 2.4% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051203.tif

------------------------------------------------------------
  Date: 20051204  |  Season: winter
  [FALLBACK] Dry day: 4.3% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051204.tif

------------------------------------------------------------
  Date

...

  Estimated   451584 blocks 
      average   0.03186077880871075  variance  0.8800679444135414
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051223.tif

------------------------------------------------------------
  Date: 20051224  |  Season: winter
  [KRIGING] 27.0% wet pixels. Proceeding with Ordinary Kriging.
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 512/512 [01:06<00:00,  7.73it/s]


  Estimated   451584 blocks 
      average   0.04046842945390029  variance  0.8141327675837877
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051224.tif

------------------------------------------------------------
  Date: 20051225  |  Season: winter
  [FALLBACK] Dry day: 4.9% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051225.tif

------------------------------------------------------------
  Date: 20051226  |  Season: winter
  [FALLBACK] Dry day: 0.6% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051226.tif

------------------------------------------------------------
  Date: 20051227  |  Season: winter
  [FALLBACK] Dry day: 0.3% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051227.tif

------------------------------------------------------------
  Date

100%|██████████| 512/512 [01:09<00:00,  7.40it/s]


  Estimated   451584 blocks 
      average   0.30811603461235054  variance  0.40086337880146417
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051228.tif

------------------------------------------------------------
  Date: 20051229  |  Season: winter
  [FALLBACK] Dry day: 3.6% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051229.tif

------------------------------------------------------------
  Date: 20051230  |  Season: winter
  [FALLBACK] Dry day: 1.9% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051230.tif

------------------------------------------------------------
  Date: 20051231  |  Season: winter
  [FALLBACK] Dry day: 1.5% wet pixels. Using Bilinear + Gaussian smooth.
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_20051231.tif
